# 实验 18 — dbt Jinja 与自定义 macro

**目标**：把 dbt 从“写 SQL 的模板引擎”用起来，看懂常见 Jinja 模式，写一个能在多个模型里复用的 macro。

## 必会的 Jinja 元素

| 模式 | 用法 |
|---|---|
| 取项目变量 | `{{ var('start_date', '2024-01-01') }}` —— 第二个参数是默认值 |
| 取环境变量 | `{{ env_var('DBT_CI', 'false') }}` |
| 当前 target | `{{ target.name }}` —— 在 dev/prod 切换行为 |
| 当前模型自引用 | `{{ this }}` —— 在 incremental 里常用 |
| 引用别的 model/source | `{{ ref('xxx') }}` / `{{ source('s','t') }}` |
| `execute` 闸门 | 只在真正 run 阶段执行，parse 阶段跳过 |

本仓库已有的 macro：
- [macros/cents_to_dollars.sql](../dbt_project/macros/cents_to_dollars.sql) —— 简单参数化 SQL
- [macros/get_active_currencies.sql](../dbt_project/macros/get_active_currencies.sql) —— `run_query` 在 parse 时查仓库，驱动 Jinja 控制流（pivot 列名动态生成等）

In [ ]:
import subprocess, os
def dbt(*a):
    r = subprocess.run(['uv','run','dbt',*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.stdout + r.stderr

# 用 `dbt run-operation` 直接调用 macro，调试 Jinja 很方便
print(dbt('run-operation','get_active_currencies'))

In [ ]:
# 在 model 里用 macro：让我们临时创建一个使用 cents_to_dollars 的分析
from pathlib import Path
p = Path('../dbt_project/analyses/example_macro_use.sql')
p.parent.mkdir(exist_ok=True)
p.write_text('''-- analysis 不会被物化，只编译成 compiled/.../example_macro_use.sql
select
    rate_date,
    currency,
    rate,
    {{ cents_to_dollars('rate * 100', precision=4) }} as rate_dollars_equiv
from {{ ref('fct_daily_rates') }}
''')
print(dbt('compile','--select','analysis:example_macro_use')[-800:])

In [ ]:
# 看 compile 后 macro 被展开成什么
compiled = Path('../dbt_project/target/compiled/dlt_dbt_sandbox/analyses/example_macro_use.sql')
if compiled.exists():
    print(compiled.read_text())
else:
    print('compiled missing—run again')

In [ ]:
# 用 --vars 传一个 model 参数（典型生产用法：CI 里覆盖 start_date）
print(dbt('run-operation','get_active_currencies','--vars','{"start_date":"2024-06-01"}')[-600:])

In [ ]:
# 演示 target.name 分支：写一个 macro 在 dev/prod 行为不同
(Path('../dbt_project/macros/env_aware_limit.sql')).write_text('''{% macro env_aware_limit() %}
  {%- if target.name == 'dev' -%}
    limit 1000
  {%- else -%}
    {# prod: no limit #}
  {%- endif -%}
{% endmacro %}
''')
print(dbt('run-operation','env_aware_limit'))
print()
print(dbt('run-operation','env_aware_limit','--target','prod'))

In [ ]:
# 清理本实验产物
import os
for f in ['../dbt_project/analyses/example_macro_use.sql','../dbt_project/macros/env_aware_limit.sql']:
    if os.path.exists(f): os.remove(f)

## 生产环境踩坑提示

- **`execute` 闸门**：用 `run_query` 取数据驱动 Jinja 时，**一定**要包 `{% if execute %}`，否则 parse 阶段查不到表会把整个项目编译挂掉。
- **macro 最好写文档**：在 `macros/_macros.yml` 里写 `description` 和参数说明，`dbt docs` 会显示。
- **不要把业务逻辑藏进 macro 太深**：macro 调试时只能看 compiled SQL；多层 macro 嵌套读起来很痛苦。深一层就行，更复杂的考虑提取到 ephemeral model。
- **`{{ target.name }}` 分支**：常见用法是 dev 只跑 1% 采样、prod 跑全量。一定要把分支逻辑放在 macro 里（一处定义），别散落到每个 model 里。
- **`var()` vs `env_var()`**：`var` 是 `dbt_project.yml` 或 `--vars` 传入的（dbt 视野内）；`env_var` 直接读 shell 环境变量，对 secrets 友好但不 hash 进 manifest，dbt 看不见变化。

## 思考题

- 用 `get_active_currencies` macro 在 model 里动态生成 pivot 列（每个货币一列），不需要硬编码 'USD','GBP'...
- 把 `cents_to_dollars` 用到 stg_ecb_rates.sql 上，看 compile 后 SQL 是什么。
- 设想团队用 dbt Cloud：env_var 怎么从 dbt Cloud 后台传进来？跟本地 `.env` 怎么共存？